In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('Airline Dataset Updated - v2.csv')
cols_to_drop = ['Passenger ID', 'Country Name', 'Pilot Name', 'Airport Name', 'First Name', 'Last Name', 'Continents', 'Airport Continent']
df = df.drop(columns=cols_to_drop)

df['Departure Date'] = pd.to_datetime(df['Departure Date'], format='mixed')
df['Departure Month'] = df['Departure Date'].dt.month
df['Departure DayOfWeek'] = df['Departure Date'].dt.dayofweek
df = df.drop(columns=['Departure Date'])

df

,Gender,Age,Nationality,Airport Country Code,Arrival Airport,Flight Status,Departure Month,Departure DayOfWeek
0,Female,62,Japan,US,CXF,On Time,6,1
1,Male,62,Nicaragua,CA,YCO,On Time,12,0
2,Male,67,Russia,FR,GNB,On Time,1,1
3,Female,71,China,CA,YND,Delayed,9,4
4,Male,21,China,US,SEE,On Time,2,4
...,...,...,...,...,...,...,...,...
98614,Male,85,China,NO,HAA,Cancelled,12,6
98615,Female,19,Russia,MG,IVA,Cancelled,10,6
98616,Male,65,Indonesia,ES,ABC,On Time,9,5
98617,Female,34,China,CI,GGN,Cancelled,10,2


In [3]:
cat_cols = ['Gender', 'Nationality', 'Airport Country Code', 'Arrival Airport']
num_cols = ['Age', 'Departure Month', 'Departure DayOfWeek']

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
X = df.drop('Flight Status', axis=1)
y = df['Flight Status']

y = LabelEncoder().fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [5]:
from sklearn.preprocessing import OrdinalEncoder
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.compose import ColumnTransformer

preprocessor_lgbm  = ColumnTransformer(transformers=[
    ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat_cols), 
    ('num', 'passthrough', num_cols)
])

preprocessor_xgb   = ColumnTransformer(transformers=[
    ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat_cols), 
    ('num', 'passthrough', num_cols)
])

lgbm = Pipeline(steps=[
    ('preprocessor', preprocessor_lgbm),
    ("lgbm", LGBMClassifier(n_jobs=-1, random_state=0))
])

xgb = Pipeline(steps=[
    ('preprocessor', preprocessor_xgb),
    ("xgb", XGBClassifier(n_jobs=-1, random_state=0))
])

In [6]:
import time

start_lgbm = time.time()
lgbm.fit(X_train, y_train)
end_lgbm = time.time()

start_xgb = time.time()
xgb.fit(X_train, y_train)
end_xgb = time.time()

print(f'it took lgbm {end_lgbm - start_lgbm:.2f} seconds to fit')
print(f'it took xgb {end_xgb - start_xgb:.2f} seconds to fit')

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000561 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 825
[LightGBM] [Info] Number of data points in the train set: 78895, number of used features: 7
[LightGBM] [Info] Start training from score -1.096536
[LightGBM] [Info] Start training from score -1.099881
[LightGBM] [Info] Start training from score -1.099424
it took lgbm 0.39 seconds to fit
it took xgb 0.64 seconds to fit


In [7]:
xgb_train_score = xgb.score(X_train, y_train)
xgb_test_score = xgb.score(X_test, y_test)

lgbm_train_score = lgbm.score(X_train, y_train)
lgbm_test_score = lgbm.score(X_test, y_test)

print('Results:')
print(f'lgbm train score: {lgbm_train_score}')
print(f'lgbm test score: {lgbm_test_score}')

print('Results:')
print(f'xgb train score: {xgb_train_score}')
print(f'xgb test score: {xgb_test_score}')

Results:
lgbm train score: 0.48398504341213006
lgbm test score: 0.33852159805313325
Results:
xgb train score: 0.5479561442423474
xgb test score: 0.3378625025349828


C:\Users\User\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\User\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [8]:
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingRandomSearchCV
import numpy as np

lgbm_param_grid = {
    'lgbm__num_leaves': np.arange(10, 150, 10),
    'lgbm__n_estimators': np.arange(100, 600, 100),
    'lgbm__learning_rate': [0.01, 0.05, 0.1],
    'lgbm__min_child_samples': [10, 20, 50],  # controls min samples per leaf
    'lgbm__reg_alpha': [0, 0.1, 0.5],         # L1 regularization
    'lgbm__reg_lambda': [0, 0.1, 0.5],        # L2 regularization
}
lgbm_search = HalvingRandomSearchCV(
    estimator=lgbm,
    param_distributions=lgbm_param_grid,
    cv=5,
    random_state=0,
    n_jobs=-1,
    min_resources=200  # adjust based on your dataset size
)

xgb_param_grid = {
    'xgb__max_depth': np.arange(3, 10, 1),
    'xgb__n_estimators': np.arange(500, 1000, 500),
    'xgb__learning_rate': [0.01, 0.05, 0.1],
    'xgb__min_child_weight': [1, 5, 10],
    'xgb__reg_alpha': [0, 0.1, 0.5],
    'xgb__reg_lambda': [0, 0.1, 0.5],
}

xgb_search = HalvingRandomSearchCV(
    estimator=xgb,
    param_distributions=xgb_param_grid,
    cv=5,
    random_state=0,
    n_jobs=-1,
    min_resources=200  # add this
)



In [9]:
import warnings
warnings.filterwarnings('ignore')

start_lgbm_grid = time.time()
lgbm_search.fit(X_train, y_train)
end_lgbm_grid = time.time()

start_xgb_grid = time.time()
xgb_search.fit(X_train, y_train)
end_xgb_grid = time.time()

print(f'it took lgbm {end_lgbm_grid - start_lgbm_grid:.2f} seconds to fit')
print(f'it took xgb {end_xgb_grid - start_xgb_grid:.2f} seconds to fit')



[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000682 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 825
[LightGBM] [Info] Number of data points in the train set: 78895, number of used features: 7
[LightGBM] [Info] Start training from score -1.096536
[LightGBM] [Info] Start training from score -1.099881
[LightGBM] [Info] Start training from score -1.099424
it took lgbm 986.35 seconds to fit
it took xgb 261.48 seconds to fit


In [10]:
# Check best params and score
print(lgbm_search.best_params_)
print(lgbm_search.best_score_)

# Evaluate on test set
print(lgbm_search.score(X_test, y_test))

# Check best params and score
print(xgb_search.best_params_)
print(xgb_search.best_score_)

# Evaluate on test set
print(xgb_search.score(X_test, y_test))

{'lgbm__reg_lambda': 0.1, 'lgbm__reg_alpha': 0.5, 'lgbm__num_leaves': np.int64(70), 'lgbm__n_estimators': np.int64(100), 'lgbm__min_child_samples': 10, 'lgbm__learning_rate': 0.05}
0.33298353909465017
0.3359359156357737
{'xgb__reg_lambda': 0.5, 'xgb__reg_alpha': 0, 'xgb__n_estimators': np.int64(500), 'xgb__min_child_weight': 5, 'xgb__max_depth': np.int64(4), 'xgb__learning_rate': 0.01}
0.33265432098765435
0.3364429121881971
